<a href="https://colab.research.google.com/github/watch-duty/radio-transcription/blob/merge_manifests/model/colabs/merge_inference_manifests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Colab to merge inference outputs of various models in different JSON files.

The google drive version of this colab (for iterating and editing) [is here](https://colab.research.google.com/drive/1W0J95R7RZ3LDWJdZf4N_oa5LeYnkqhgX). Once you have a unit of edit completed, you can check-in the drive version into [this github directory](https://github.com/watch-duty/radio-transcription/tree/main/model/colabs).


In [1]:
#@title Imports
import json
import os

root_manifest_directory = "/usr/local/google/home/varungulshan/radio-transcription/model/data/inference_manifests"  #@param
# The ground truth field will be picked up from the first json file.
jsons_and_eval_ids_to_merge = [
    ("playground_parakeet_and_canary_flash_and_chirp_with_context.jsonl",
     ["parakeet-tdt-06b-v2", "canary-1b-flash", "chirp_v3_with_context"]),
    ("playground_parakeet_and_canary_flash_and_chirp.jsonl",
     ["chirp_v3_no_context"]),
    ("playground_parakeet_and_canary_flash_and_gemma3n_e2b_it.jsonl",
     ["gemma3n"]),
    ]
merged_output = "playground_all_models.jsonl" #@param



In [32]:
#@title Helper functions

def run_json_merge(jsons_and_postfixes, output_json):
  gt_text_field = "text"
  prediction_field_prefix = "pred_text_"
  output_json = os.path.join(root_manifest_directory, output_json)

  file_handles = []
  postfixes_per_json = []
  for json_file, postfixes in jsons_and_postfixes:
    json_file = os.path.join(root_manifest_directory, json_file)
    file_handles.append(open(json_file, 'r', encoding='utf-8'))
    postfixes_per_json.append(postfixes)

  with open(output_json, 'w', encoding='utf-8') as out:
    # Loop through each line simultanesoulys across all the JSON files.
    while True:
      lines = []
      for f in file_handles:
        lines.append(f.readline().strip())

      if lines[0] == "":
        for line in lines[1:]:
          assert line == "", "Expect all jsonl to have the same number of lines"
        break

      # Setup the base data row object from the first line, by copying over
      # all the attributes except those that being with 'pred_text' as those
      # correspond to various model outputs. This also copies over the
      # ground truth that is stored in 'text'.
      output_row = {}
      first_output_row = json.loads(lines[0])
      for key, value in first_output_row.items():
        if not key.startswith(prediction_field_prefix):
          output_row[key] = first_output_row[key]

      # Loop through each json line and its corresponding postfixes and
      # load the prediction outputs into the output_data_row. Make assertions
      # along the way to make sure you are joining the right data.
      for line, postfixes in zip(lines, postfixes_per_json):
        data_row = json.loads(line)
        assert data_row["audio_filepath"] == output_row["audio_filepath"], "Audio filepaths should be in same order for all jsons"
        assert data_row["offset"] == output_row["offset"], "Audio offsets should be identical across jsons"
        assert data_row["duration"] == output_row["duration"], "Audio duration should be identical across jsons"
        for postfix in postfixes:
          pred_text_field = prediction_field_prefix + postfix
          output_row[pred_text_field] = data_row[pred_text_field]

      # Write out the output_row which has all the merged outputs by now.
      out.write(json.dumps(output_row) + '\n')


  pprint_jsons = ""
  for tuple in jsons_and_postfixes:
    pprint_jsons += str(tuple) + "\n"
  print(f"Successfully merged:\n{pprint_jsons} into {output_json}")

In [33]:
run_json_merge(jsons_and_eval_ids_to_merge, merged_output)

Successfully merged:
('playground_parakeet_and_canary_flash_and_chirp_with_context.jsonl', ['parakeet-tdt-06b-v2', 'canary-1b-flash', 'chirp_v3_with_context'])
('playground_parakeet_and_canary_flash_and_chirp.jsonl', ['chirp_v3_no_context'])
('playground_parakeet_and_canary_flash_and_gemma3n_e2b_it.jsonl', ['gemma3n'])
 into /usr/local/google/home/varungulshan/radio-transcription/model/data/inference_manifests/playground_all_models.jsonl
